# Tidy Data {#sec-data-tidy}

## Introduction

In this chapter, you will learn a consistent way to organise your data in Python using the principle known as *tidy data*. Tidy data is not appropriate for everything, but for a lot of analysis and a lot of tabular data it will be what you need. Getting your data into this format requires some work up front, but that work pays off in the long term. Once you have tidy data, you will spend much less time munging data from one representation to another, allowing you to spend more time on the data questions you care about.

In this chapter, you'll first learn the definition of tidy data and see it applied to simple toy dataset. Then we'll dive into the main tool you'll use for tidying data: unpivoting. Unpivoting allows you to change wide data into a long format without altering any underlying values. We'll finish up with a discussion of usefully untidy data, and how you can create it if needed.

If you particularly enjoy this chapter and want to learn more about the underlying theory, you can learn more in the [Tidy Data](https://www.jstatsoft.org/article/view/v059i10) paper published in the Journal of Statistical Software.


In [ ]:
# remove cell
import matplotlib.pyplot as plt
import matplotlib_inline.backend_inline

# Plot settings
plt.style.use("https://github.com/aeturrell/python4DS/raw/main/plot_style.txt")
matplotlib_inline.backend_inline.set_matplotlib_formats("svg")

### Prerequisites

This chapter will use the **polars** data analysis package.

## Tidy Data

There are three interrelated features that make a dataset tidy:

1.  Each variable is a column; each column is a variable.
2.  Each observation is row; each row is an observation.
3.  Each value is a cell; each cell is a single value.

The figure below shows this:

![](https://d33wubrfki0l68.cloudfront.net/6f1ddb544fc5c69a2478e444ab8112fb0eea23f8/91adc/images/tidy-1.png)

Why ensure that your data is tidy? There are two main advantages:

1.  There's a general advantage to picking one consistent way of storing data.
    If you have a consistent data structure, it's easier to learn the tools that work with it because they have an underlying uniformity. Some tools, for example data visualisation package **seaborn**, are designed with tidy data in mind.

2.  There's a specific advantage to placing variables in columns because it allows you to take advantage of **polars**' vectorised operations (operations that are more efficient).


Tidy data aren't going to be appropriate *every* time and in every case, but they're a really, really good default for tabular data. Once you use it as your default, it's easier to think about how to perform subsequent operations.

Having said that tidy data are great, they are, but one of **polars**' advantages relative to other data analysis libraries is that it isn't *too* tied to tidy data and can navigate awkward non-tidy data manipulation tasks happily too.

There are two common problems you find in data that are ingested that make them not tidy:

1. A variable might be spread across multiple columns.
2. An observation might be scattered across multiple rows.

For the former, we need to "melt" the wide data, with multiple columns, into long data.

For the latter, we need to unstack or pivot the multiple rows into columns (ie go from long to wide.)

We'll see both below.

## Tools to Make Data Tidy with **polars**

### Unpivot

`unpivot()` can help you go from "wider" data to "longer" data, and is a *really* good one to remember.

![](https://pandas.pydata.org/docs/_images/reshaping_melt.png)

Here's an example of it in action:

In [ ]:
import polars as pl

df = pl.DataFrame(
    {
        "first": ["John", "Mary"],
        "last": ["Doe", "Bo"],
        "job": ["Nurse", "Economist"],
        "height": [5.5, 6.0],
        "weight": [130, 150],
    }
)
print("\n Wide DataFrame: ")
print(df)
print("\n Unpivoted (Long) DataFrame: ")
df.unpivot(
    index=["first", "last"],
    variable_name="quantity",
    value_name="value",
    on=["height", "weight"],
)

::: {.callout-tip title="Exercise"}
Perform a `unpivot()` that uses `job` as the id instead of `first` and `last`.
:::

How does this relate to tidy data? Sometimes you'll have a variable spread over multiple columns that you want to turn tidy. Let's look at this example that uses cases of [tuburculosis from the World Health Organisation](https://www.who.int/teams/global-tuberculosis-programme/data).

First let's open the data and look at the top of the file.

In [ ]:
df_tb = pl.read_parquet(
    "https://github.com/aeturrell/python4DS/raw/refs/heads/main/data/who_tb_cases.parquet"
)
df_tb.head()

You can see that we have two columns for a single variable, year. Let's now unpivot this.

In [ ]:
df_tb.unpivot(
    index=["country"],
    variable_name="year",
    value_name="cases",
    on=["1999", "2000"],
)

We now have one observation per row, and one variable per column: tidy!

### Reshaping Wide Data with Multiple Variables

Sometimes you have multiple variables spread across columns in a wide format. Here's how to handle that in polars:

In [ ]:
import numpy as np

df = pl.DataFrame(
    {
        "A1970": ["a", "b", "c"],
        "A1980": ["d", "e", "f"],
        "B1970": [2.5, 1.2, 0.7],
        "B1980": [3.2, 1.3, 0.1],
        "X": np.random.randn(3),
        "id": [0, 1, 2],
    }
)
df

To reshape this data, we first unpivot, then extract the year and variable:

In [ ]:
df_long = df.unpivot(
    index=["id", "X"],
    variable_name="variable_year",
    value_name="value",
    on=["A1970", "A1980", "B1970", "B1980"],
)

df_long = df_long.with_columns(
    [
        pl.col("variable_year").str.slice(0, 1).alias("variable"),
        pl.col("variable_year").str.slice(1, 4).cast(pl.Int64).alias("year"),
    ]
)

df_final = df_long.drop("variable_year").pivot(
    index=["id", "X", "year"],
    on="variable",
    values="value",
    aggregate_function="first",
)

df_final.sort(["id", "year"])

### Reshaping Multi-Column Data

Polars uses unpivot() and pivot() for reshaping. 

In **polars**, reshaping between long and wide tabular formats is performed seamlessly using `unpivot()` (to go long) and `pivot()` (to go wide), without needing row indices or complex MultiIndexes.

![](https://pandas.pydata.org/docs/_images/reshaping_stack.png)

To go back to wider format (unstack)

![](https://pandas.pydata.org/docs/_images/reshaping_unstack.png)

Here's how to work with hierarchical data:

In [ ]:
# Create a DataFrame with multiple columns
df = pl.DataFrame(
    {
        "first": ["bar", "bar", "baz", "baz", "foo", "foo", "qux", "qux"],
        "second": ["one", "two", "one", "two", "one", "two", "one", "two"],
        "A": np.random.randn(8),
        "B": np.random.randn(8),
    }
)
df

To go to a longer format (stack):

In [ ]:
# Unpivot to stack the A and B columns
df_stacked = df.unpivot(
    index=["first", "second"],
    variable_name="variable",
    value_name="value",
    on=["A", "B"],
)
df_stacked

then to wider format (unstack)

In [ ]:
# Pivot back to wider format
df_unstacked = df_stacked.pivot(
    index=["first", "second"],
    on="variable",
    values="value",
    aggregate_function="first",
)
df_unstacked

### Pivoting from Long to Wide

pivot() helps you to sort out data in which a single observation is scattered over multiple rows.

![](https://pandas.pydata.org/docs/_images/reshaping_pivot.png)

Here's an example dataframe where observations are spread over multiple rows:

In [ ]:
df_tb_cp = pl.read_parquet(
    "https://github.com/aeturrell/python4DS/raw/refs/heads/main/data/who_tb_case_and_pop.parquet"
)
df_tb_cp.head()

You see that we have, for each year-country, "case" and "population" in different rows.

Now let's pivot this to see the difference:

In [ ]:
pivoted = df_tb_cp.pivot(
    index=["country", "year"],
    on="type",
    values="count",
    aggregate_function="first",
).sort(["country", "year"])
pivoted

Pivots are especially useful for time series data, where operations like `shift()` are typically applied assuming that an entry in one row follows (in time) from the one above. When we do `shift()` we often want to shift a single variable in time, but if a single observation (in this case a date) is over multiple rows, the timing goes awry. Let's see an example.

In [ ]:
import numpy as np
import polars as pl

dates = list(
    pl.date_range(
        start=pl.date(2000, 1, 31),
        end=pl.date(2000, 10, 31),
        interval="1mo",
        eager=True,
    )
)

data = {
    "value": np.random.randn(20),
    "variable": ["A"] * 10 + ["B"] * 10,
    "date": dates + dates,
}
df = pl.DataFrame(data)
df.sample(5)

If we just run `shift()` on the above, it's going to shift variable B's and A's together even though they overlap in time and are different variables. So we pivot to a wider format to shift safely:

In [ ]:
df_pivoted = df.pivot(
    index="date",
    on="variable",
    values="value",
    aggregate_function="first",
).with_columns(
    [
        pl.col("A").shift(1),
        pl.col("B").shift(1),
    ]
)
df_pivoted

::: {.callout-tip title="Exercise"}
Why is the first entry `null`?
:::


::: {.callout-tip title="Exercise"}
Perform a `pivot()` that applies to both the `variable` and `category` columns in the example from above where category is defined such that `df = df.with_columns(category=pl.Series(np.random.choice(["type1", "type2", "type3", "type4"], 20)))`. (Hint: pass multiple column names to the `on` parameter as a list.)
:::